# Sprint 2 — MegaMarket: ¿qué clientes entran en la ruta de esta semana?

Jacobo Argote, Andrés Cardona, Miguel Valencia y Juan José Gil

##**CÓDIGO GENERADO CON CLAUDE**

## De dónde venimos (Sprint 1) y qué agregamos en este sprint

En el Sprint 1 construimos A* para encontrar la ruta más corta entre dos
puntos, y `recorrido_completo` para que el camión visite **todos** los
clientes de una ruta (Poblado o Valle de Aburrá) usando "vecino más
cercano". El problema: `clientes_poblado` y `clientes_valle_aburra` eran
listas **fijas** — las mismas 8 tiendas y los mismos 4 CEDIs, todas las
semanas, sin importar si de verdad necesitaban una entrega.

En la vida real eso no tiene sentido: no todos los clientes necesitan
mercancía cada semana. Antes de correr A* hace falta **decidir quién
entra en la ruta**. Para eso construimos un **sistema experto** con la
estructura clásica vista en clase:

1. **Base de conocimiento**: lo que MegaMarket sabe de forma *estable*
   (reglas generales del negocio, atributos fijos de cada cliente).
2. **Memoria de trabajo**: los datos *de esta semana en particular* para
   cada cliente.
3. **Motor de inferencia**: el "razonador" que combina las dos piezas
   anteriores y concluye si el cliente entra en la ruta, con qué
   prioridad, y **por qué**.

La salida del motor de inferencia reemplaza las listas fijas
`clientes_poblado` / `clientes_valle_aburra`. El resto del Sprint 1 (A*,
la heurística, los grafos de cada ruta) **no cambia**.

## Parte 0 — Lo heredado del Sprint 1 (sin cambios de lógica)

Coordenadas, heurística (haversine), los grafos de cada ruta y A* quedan
exactamente igual que en el sprint anterior.

In [35]:
from heapq import heappush, heappop
from math import radians, sin, cos, asin, sqrt

# =============================================================================
# 1. COORDENADAS (lat, lon) de cada punto real de la operación
# =============================================================================
COORDENADAS = {
    "Bodega Olaya Herrera": (6.2197, -75.5905),

    # --- Accesos viales a cada zona ---
    "Acceso Poblado (Aguacatala)": (6.1935, -75.5790),
    "Acceso Sur (Itagüí Centro)": (6.1720, -75.6120),

    # --- Ruta Poblado: 8 clientes ---
    "Fruver del Río": (6.2218, -75.5745),
    "Fruver Visitación": (6.2013, -75.5598),
    "Five Sisters": (6.2077, -75.5658),
    "Supermercado Merka 10": (6.2093, -75.5688),
    "Hotel Four Points": (6.2029, -75.5722),
    "Mercados Verona": (6.2046, -75.5605),
    "EAFIT La Bodeguita": (6.2006, -75.5783),
    "Redetrans": (6.2112, -75.5628),

    # --- Ruta Valle de Aburrá (Sur): 4 clientes ---
    "CEDI Vaquita Itagüí": (6.1735, -75.6152),
    "CEDI Euro La Estrella": (6.1448, -75.6198),
    "CEDI Merkepaisa": (6.1618, -75.6062),
    "CEDI Supermú": (6.1548, -75.6122),

    # --- Destino final compartido por las dos rutas ---
    "Plataforma Logística": (6.1976, -75.5807),
}

# =============================================================================
# 2. HEURÍSTICA h(n): distancia en línea recta (haversine), igual al Sprint 1
# =============================================================================
def heuristica(nodo, destino):
    """Distancia en línea recta entre dos lugares, en kilómetros."""
    lat1, lon1 = COORDENADAS[nodo]
    lat2, lon2 = COORDENADAS[destino]
    lat1, lon1, lat2, lon2 = map(radians, [lat1, lon1, lat2, lon2])
    d_lat = lat2 - lat1
    d_lon = lon2 - lon1
    aux = sin(d_lat / 2) ** 2 + cos(lat1) * cos(lat2) * sin(d_lon / 2) ** 2
    return 2 * 6371.0 * asin(sqrt(aux))  # 6371 km = radio de la Tierra


# =============================================================================
# 3. GRAFOS DE CADA RUTA (igual al Sprint 1)
# =============================================================================
def conectar(red, a, b, costo):
    """Agrega una conexión en AMBOS sentidos (a->b y b->a)."""
    red.setdefault(a, []).append((b, costo))
    red.setdefault(b, []).append((a, costo))


red_poblado = {}
conectar(red_poblado, "Bodega Olaya Herrera", "Acceso Poblado (Aguacatala)", 3.2)
conectar(red_poblado, "Acceso Poblado (Aguacatala)", "Fruver del Río", 3.2)
conectar(red_poblado, "Acceso Poblado (Aguacatala)", "Fruver Visitación", 2.3)
conectar(red_poblado, "Acceso Poblado (Aguacatala)", "Five Sisters", 2.2)
conectar(red_poblado, "Acceso Poblado (Aguacatala)", "Supermercado Merka 10", 2.1)
conectar(red_poblado, "Acceso Poblado (Aguacatala)", "Hotel Four Points", 1.3)
conectar(red_poblado, "Acceso Poblado (Aguacatala)", "Mercados Verona", 2.4)
conectar(red_poblado, "Acceso Poblado (Aguacatala)", "EAFIT La Bodeguita", 0.8)
conectar(red_poblado, "Acceso Poblado (Aguacatala)", "Redetrans", 2.7)

red_valle_aburra = {}
conectar(red_valle_aburra, "Bodega Olaya Herrera", "Acceso Sur (Itagüí Centro)", 5.8)
conectar(red_valle_aburra, "Acceso Sur (Itagüí Centro)", "CEDI Vaquita Itagüí", 0.4)
conectar(red_valle_aburra, "Acceso Sur (Itagüí Centro)", "CEDI Euro La Estrella", 3.2)
conectar(red_valle_aburra, "Acceso Sur (Itagüí Centro)", "CEDI Merkepaisa", 1.3)
conectar(red_valle_aburra, "Acceso Sur (Itagüí Centro)", "CEDI Supermú", 1.9)

conectar(
    red_poblado, "Acceso Poblado (Aguacatala)", "Plataforma Logística",
    round(heuristica("Acceso Poblado (Aguacatala)", "Plataforma Logística"), 1),
)
conectar(
    red_valle_aburra, "Acceso Sur (Itagüí Centro)", "Plataforma Logística",
    round(heuristica("Acceso Sur (Itagüí Centro)", "Plataforma Logística"), 1),
)

# Universo TOTAL de clientes de cada ruta (el catálogo completo). Ya no se le
# entregan directo a recorrido_completo: ahora el sistema experto decide,
# cada semana, cuál subconjunto de este universo realmente sale a reparto.
clientes_poblado = [
    "Fruver del Río", "Fruver Visitación", "Five Sisters",
    "Supermercado Merka 10", "Hotel Four Points", "Mercados Verona",
    "EAFIT La Bodeguita", "Redetrans",
]
clientes_valle_aburra = [
    "CEDI Vaquita Itagüí", "CEDI Euro La Estrella",
    "CEDI Merkepaisa", "CEDI Supermú",
]

UMBRAL_CERCANIA_KM = 1.5


def conectar_clientes_cercanos(red, clientes, umbral=UMBRAL_CERCANIA_KM):
    for i in range(len(clientes)):
        for j in range(i + 1, len(clientes)):
            a, b = clientes[i], clientes[j]
            distancia = heuristica(a, b)
            if distancia < umbral:
                conectar(red, a, b, round(distancia, 2))


conectar_clientes_cercanos(red_poblado, clientes_poblado)
conectar_clientes_cercanos(red_valle_aburra, clientes_valle_aburra)


# =============================================================================
# 4. A* (A-Star) — igual al Sprint 1
# =============================================================================
def a_estrella(red, origen, destino):
    cola = [(heuristica(origen, destino), 0, [origen])]
    visitados = set()

    while cola:
        f, g, camino = heappop(cola)
        nodo_actual = camino[-1]

        if nodo_actual == destino:
            return camino, g

        if nodo_actual not in visitados:
            visitados.add(nodo_actual)
            for vecino, costo in red[nodo_actual]:
                nuevo_g = g + costo
                nuevo_f = nuevo_g + heuristica(vecino, destino)
                heappush(cola, (nuevo_f, nuevo_g, camino + [vecino]))

    return None, None

## Parte 0.1 — `recorrido_completo`: un solo cambio (agregar prioridad)

La firma de la función gana un parámetro opcional `prioridad`, para poder
cumplir la regla *"si hay queja urgente, visitar primero"*. Los clientes
se agrupan en **tandas** (ALTA, MEDIA, BAJA); el camión agota TODA la
tanda de mayor prioridad (con vecino más cercano *dentro* de esa tanda)
antes de pasar a la siguiente. Si no se pasa prioridad, todos quedan en
una sola tanda y el comportamiento es **exactamente el del Sprint 1**.

In [36]:
# =============================================================================
# 5. RECORRIDO COMPLETO DE REPARTO (Sprint 1 + tandas de prioridad)
# =============================================================================
def recorrido_completo(red, origen, clientes, destino_final=None, prioridad=None):
    """Recorrido de reparto: sale de 'origen', visita cada cliente una sola
    vez y termina en 'destino_final'.

    'prioridad' es un diccionario opcional {cliente: "ALTA"/"MEDIA"/"BAJA"}
    que viene del sistema experto. Los clientes se atienden en tandas: se
    agota TODA la tanda de mayor prioridad (con vecino más cercano dentro
    de la tanda) antes de pasar a la siguiente. Si no se indica prioridad,
    todos quedan en una sola tanda ("NORMAL") y el resultado es idéntico
    al del Sprint 1.
    """
    if destino_final is None:
        destino_final = origen
    if prioridad is None:
        prioridad = {c: "NORMAL" for c in clientes}

    orden_tandas = ["ALTA", "MEDIA", "BAJA", "NORMAL"]
    tandas = {
        nivel: [c for c in clientes if prioridad.get(c, "NORMAL") == nivel]
        for nivel in orden_tandas
    }

    paradas = [origen]
    orden_clientes = []
    posicion_actual = origen
    distancia_total = 0

    # Recorremos cada tanda en orden de prioridad (ALTA -> MEDIA -> BAJA/NORMAL)
    for nivel in orden_tandas:
        pendientes = tandas[nivel]

        # Dentro de cada tanda, exactamente el mismo "vecino más cercano" del Sprint 1
        while pendientes:
            mejor_cliente, mejor_camino, mejor_costo = None, None, float("inf")
            for cliente in pendientes:
                camino, costo = a_estrella(red, posicion_actual, cliente)
                if costo < mejor_costo:
                    mejor_cliente, mejor_camino, mejor_costo = cliente, camino, costo

            paradas.extend(mejor_camino[1:])
            orden_clientes.append(mejor_cliente)
            distancia_total += mejor_costo
            posicion_actual = mejor_cliente
            pendientes.remove(mejor_cliente)

    # Último tramo: hacia destino_final
    camino_final, costo_final = a_estrella(red, posicion_actual, destino_final)
    paradas.extend(camino_final[1:])
    distancia_total += costo_final

    return paradas, distancia_total, orden_clientes

## Parte 1 — Base de conocimiento

Igual que antes, junta lo que MegaMarket sabe de forma **estable**. La
diferencia con la primera versión: ya no asumimos que `stock_actual` es
un número exacto disponible en todos los puntos de venta. En su lugar,
el conocimiento del negocio se apoya en dos cosas reales:

- **La rotación del producto en cada punto de venta** (alta / media /
  baja) — la conoce el mercaderista, y es un atributo relativamente
  **estable** de cada cliente (no cambia cada semana).
- **Las políticas comerciales de MegaMarket** (los umbrales de tiempo,
  de flete, y la ventana de entrega comprometida).

In [37]:
# =============================================================================
# 6. BASE DE CONOCIMIENTO — reglas generales, hechos y relaciones (estable)
# =============================================================================
BASE_CONOCIMIENTO = {

    # --- Reglas generales de negocio (umbrales que usan R2 y R4) ---
    "umbrales": {
        "dias_sin_entrega": 15,        # días sin entrega para activar el respaldo (R2)
        "flete_sobre_factura": 0.04,   # 4% — tope de flete sobre la factura (R4)
    },

    # --- Política de servicio (H1): no filtra clientes, solo documenta el compromiso ---
    "politicas": {
        "ventana_entrega_horas": (48, 72),
    },

    # --- Hechos y relaciones: atributos FIJOS de cada cliente ---
    # (a diferencia de la visita del mercaderista o las quejas, la ruta y la
    # rotación del producto en ese punto no cambian cada semana)
    "clientes": {
        # Ruta Poblado
        "Fruver del Río":          {"ruta": "Poblado", "rotacion_producto": "alta"},
        "Fruver Visitación":       {"ruta": "Poblado", "rotacion_producto": "media"},
        "Five Sisters":            {"ruta": "Poblado", "rotacion_producto": "media"},
        "Supermercado Merka 10":   {"ruta": "Poblado", "rotacion_producto": "alta"},
        "Hotel Four Points":       {"ruta": "Poblado", "rotacion_producto": "baja"},
        "Mercados Verona":         {"ruta": "Poblado", "rotacion_producto": "media"},
        "EAFIT La Bodeguita":      {"ruta": "Poblado", "rotacion_producto": "alta"},
        "Redetrans":               {"ruta": "Poblado", "rotacion_producto": "baja"},

        # Ruta Valle de Aburrá (Sur)
        "CEDI Vaquita Itagüí":     {"ruta": "Valle de Aburrá", "rotacion_producto": "alta"},
        "CEDI Euro La Estrella":   {"ruta": "Valle de Aburrá", "rotacion_producto": "media"},
        "CEDI Merkepaisa":         {"ruta": "Valle de Aburrá", "rotacion_producto": "media"},
        "CEDI Supermú":            {"ruta": "Valle de Aburrá", "rotacion_producto": "baja"},
    },

    # --- Relación: destino final compartido por todas las rutas ---
    "destino_final": "Plataforma Logística",
}

## Parte 2 — Las reglas de negocio, una por una

**R1, R2 y R3 son reglas de inclusión** (basta con que se cumpla una).
**R4 y R5 son restricciones duras (veto)**: si cualquiera se activa,
anula la inclusión sin importar qué tanto la justificaran R1/R2/R3.
**R6 es la regla por defecto**. **H1** es una política de servicio, no
una regla condicional.

| # | Regla | Efecto |
|---|-------|--------|
| **R1** | SI el mercaderista, en su visita más reciente, reporta que el stock de la referencia está bajo, ENTONCES incluir al cliente. La prioridad depende de la rotación del producto en ese punto: rotación ALTA → prioridad ALTA, MEDIA → MEDIA, BAJA → BAJA. | Inclusión |
| **R2** | SI el punto de venta NO tiene visita reciente del mercaderista Y han pasado más de 15 días desde la última entrega Y la rotación del producto ahí es MEDIA o ALTA, ENTONCES incluir preventivamente. Es un respaldo — solo aplica si NO hubo visita; si el mercaderista sí visitó y dijo que el stock está bien, esa observación directa manda por encima del tiempo. | Inclusión, prioridad **BAJA** |
| **R3** | SI el cliente reportó una queja urgente, ENTONCES incluirlo en la ruta y darle prioridad. | Inclusión, prioridad **ALTA** |
| **R4** | SI el valor del flete es mayor al 4% del valor de la factura del pedido, ENTONCES NO se despacha. | Veto |
| **R5** | SI el punto de venta NO ha actualizado el precio de venta al público (PVP) de la referencia según la lista vigente de MegaMarket, ENTONCES NO se despacha. | Veto |
| **R6** | SI ninguna de R1, R2 o R3 se cumple, ENTONCES el cliente no entra en la ruta de esta semana. | Exclusión por defecto |

**H1 — Política de entrega:** todo pedido incluido en la ruta debe
entregarse entre 48 y 72 horas después de generada la orden. Se
documenta como un hecho de la base de conocimiento (el compromiso de
servicio bajo el que opera la ruta), no como una regla que decide *quién*
entra — `recorrido_completo` hoy no modela tiempos ni calendario, así
que por ahora queda registrada pero sin verificación computacional.

**Jerarquía / resolución de conflictos:** primero se evalúan R1-R3; si
ninguna aplica, R6 excluye directo. Si el cliente sí es candidato, recién
ahí se revisan R4 y R5 — cualquiera de las dos (o ambas) anula la
inclusión, incluso si la candidatura venía de una queja urgente (R3): la
rentabilidad y el cumplimiento de precios protegen la operación de
MegaMarket por encima del motivo de inclusión.

## Parte 3 — Memoria de trabajo

Los datos *de esta semana en particular* para cada cliente ya no se
escriben a mano dentro del notebook: viven en un CSV externo
(`memoria_trabajo_logistica.csv`), con **una fila por cliente y una
columna por cada dato que necesitan las reglas** — exactamente los
mismos campos que antes iban en el diccionario `MEMORIA_TRABAJO`. Cuando
no hubo visita del mercaderista, la celda dice explícitamente **"No hubo
visita"** en vez de quedar vacía -- así el dato ausente queda documentado
en la tabla, no escondido en una celda en blanco.

Esto separa dos cosas que cambian a ritmos distintos: el *código* (las
reglas, el motor de inferencia, A*) casi no cambia semana a semana; los
*datos* de la operación sí, y ahora se actualizan editando el CSV — sin
tocar una sola línea de Python. Es también más parecido a cómo funcionaría
en producción, donde esta tabla saldría de un reporte de mercaderistas
o de un export del ERP.

**Antes de correr la siguiente celda:** sube `memoria_trabajo_logistica.csv`
a esta sesión de Colab (ícono de carpeta 📁 en la barra lateral izquierda
→ botón "Subir al almacenamiento de la sesión"), o móntalo desde Google
Drive si prefieres que persista entre sesiones.

In [38]:
# =============================================================================
# 7. MEMORIA DE TRABAJO — datos de ESTA semana (leídos desde CSV)
# =============================================================================
import pandas as pd

# sep=None + engine="python" detecta solo si el CSV viene separado por comas
# o por punto y coma (Excel en español/Latinoamérica exporta con ";" porque
# usa la coma como separador decimal). encoding="utf-8-sig" quita el BOM
# invisible que Excel agrega al guardar como "CSV UTF-8". Así la celda no
# se rompe sin importar cómo lo haya guardado la última persona que lo editó.
memoria_df = pd.read_csv("memoria_trabajo_logistica.csv", sep=None, engine="python", encoding="utf-8-sig")


def a_booleano(valor):
    """Convierte 'Sí' / 'No' del CSV a True / False."""
    return str(valor).strip().lower() == "sí"


def interpretar_stock_reportado(valor):
    """La columna 'mercaderista_reporta_stock_bajo' tiene TRES estados
    posibles, no dos: 'Sí' (reportó stock bajo), 'No' (visitó y el stock
    está bien) o 'No hubo visita' (no hay observación -- ahí es donde
    entra en juego la regla R2, el respaldo por falta de visibilidad).
    """
    valor = str(valor).strip().lower()
    if valor == "sí":
        return True
    if valor == "no":
        return False
    return None  # "No hubo visita" (o cualquier otro texto que no sea Sí/No)


MEMORIA_TRABAJO = {}
for _, fila in memoria_df.iterrows():
    MEMORIA_TRABAJO[fila["cliente"]] = {
        "visita_mercaderista_disponible": a_booleano(fila["visita_mercaderista_disponible"]),
        "mercaderista_reporta_stock_bajo": interpretar_stock_reportado(fila["mercaderista_reporta_stock_bajo"]),
        "dias_desde_ultima_entrega": int(fila["dias_desde_ultima_entrega"]),
        "queja_urgente": a_booleano(fila["queja_urgente"]),
        "precio_actualizado_en_punto_de_venta": a_booleano(fila["precio_actualizado_en_punto_de_venta"]),
        "valor_factura_pedido": int(fila["valor_factura_pedido"]),
        "costo_flete_estimado": int(fila["costo_flete_estimado"]),
    }

# Vista rápida de lo que se cargó, para confirmar que el CSV subió bien
memoria_df

,cliente,visita_mercaderista_disponible,mercaderista_reporta_stock_bajo,dias_desde_ultima_entrega,queja_urgente,precio_actualizado_en_punto_de_venta,valor_factura_pedido,costo_flete_estimado
0,Fruver del Río,Sí,Sí,10,No,Sí,850000,50000
1,Fruver Visitación,Sí,No,20,No,Sí,600000,15000
2,Five Sisters,No,No hubo visita,18,No,Sí,700000,20000
3,Supermercado Merka 10,Sí,No,8,Sí,Sí,500000,15000
4,Hotel Four Points,Sí,No,5,No,Sí,1200000,18000
5,Mercados Verona,Sí,Sí,22,No,Sí,300000,10000
6,EAFIT La Bodeguita,No,No hubo visita,6,Sí,Sí,900000,20000
7,Redetrans,Sí,Sí,12,No,No,400000,13000
8,CEDI Vaquita Itagüí,Sí,Sí,14,No,Sí,2500000,40000
9,CEDI Euro La Estrella,No,No hubo visita,25,No,Sí,1800000,95000


## Parte 4 — Motor de inferencia

Sigue siendo un motor de **encadenamiento hacia adelante**, evaluado en
una sola pasada por cliente (misma honestidad de alcance que discutimos
para 2-opt vs. fuerza bruta: no hay dependencias entre clientes, así que
no hace falta un motor con iteración hasta punto fijo tipo RETE).

Para cada cliente:

1. Junta los **hechos fijos** (`rotacion_producto`, `ruta`) con los
   **hechos de esta semana** (visita del mercaderista, días sin entrega,
   queja, PVP, factura, flete).
2. Evalúa R1, R2 y R3. Si el mercaderista reportó stock bajo, la
   prioridad sale de la rotación del producto (`MAPA_ROTACION_PRIORIDAD`).
3. Si ninguna de las tres se cumple, aplica R6 y termina.
4. Si el cliente es candidato, revisa R4 y R5 — cualquiera que se
   cumpla anula la inclusión.
5. Devuelve la conclusión junto con las razones que la sustentan.

In [39]:
# =============================================================================
# 8. MOTOR DE INFERENCIA
# =============================================================================
NIVEL_PRIORIDAD = {"ALTA": 3, "MEDIA": 2, "BAJA": 1}

# La rotación del producto en el punto de venta define la prioridad de R1
MAPA_ROTACION_PRIORIDAD = {"alta": "ALTA", "media": "MEDIA", "baja": "BAJA"}


def prioridad_mas_alta(actual, nueva):
    """Se queda con la prioridad más alta entre 'actual' y 'nueva'."""
    if actual is None or NIVEL_PRIORIDAD[nueva] > NIVEL_PRIORIDAD[actual]:
        return nueva
    return actual


def motor_inferencia(cliente, base_conocimiento, memoria_trabajo):
    """Aplica la base de conocimiento (reglas) sobre los hechos de UN
    cliente (memoria de trabajo) y devuelve la conclusión: si entra en la
    ruta de esta semana, con qué prioridad, y por qué.
    """
    hechos_fijos = base_conocimiento["clientes"][cliente]   # ruta, rotacion_producto
    hechos_semana = memoria_trabajo[cliente]                # visita, dias, queja, PVP, factura, flete
    umbrales = base_conocimiento["umbrales"]

    razones = []
    prioridad = None

    # --- R1: el mercaderista vio, en su última visita, que el stock está bajo ---
    if hechos_semana["visita_mercaderista_disponible"] and hechos_semana["mercaderista_reporta_stock_bajo"]:
        rotacion = hechos_fijos["rotacion_producto"]
        prioridad_r1 = MAPA_ROTACION_PRIORIDAD[rotacion]
        razones.append(
            f"R1 - El mercaderista reportó stock bajo en su visita "
            f"(rotación {rotacion} -> prioridad {prioridad_r1})"
        )
        prioridad = prioridad_mas_alta(prioridad, prioridad_r1)

    # --- R2: respaldo por falta de visibilidad (sin visita + mucho tiempo + rotación relevante) ---
    # OJO: si SÍ hubo visita reciente, esta regla NO aplica -- la observación
    # directa del mercaderista manda sobre cualquier estimación por tiempo.
    if (not hechos_semana["visita_mercaderista_disponible"]
            and hechos_semana["dias_desde_ultima_entrega"] > umbrales["dias_sin_entrega"]
            and hechos_fijos["rotacion_producto"] in ("media", "alta")):
        razones.append(
            f"R2 - Sin visita reciente del mercaderista, "
            f"{hechos_semana['dias_desde_ultima_entrega']} días sin entrega "
            f"(> {umbrales['dias_sin_entrega']}), rotación {hechos_fijos['rotacion_producto']}"
        )
        prioridad = prioridad_mas_alta(prioridad, "BAJA")

    # --- R3: queja urgente reportada ---
    if hechos_semana["queja_urgente"]:
        razones.append("R3 - Queja urgente reportada: debe visitarse primero")
        prioridad = prioridad_mas_alta(prioridad, "ALTA")

    # --- R6: regla por defecto (ninguna regla de inclusión se cumplió) ---
    if not razones:
        return {
            "incluir_en_ruta": False,
            "prioridad": None,
            "razones": ["R6 - Por defecto: no cumple ningún criterio de inclusión esta semana"],
        }

    # --- R4 y R5: restricciones duras (veto). Solo importan si el cliente ya es candidato ---
    vetos = []

    ratio_flete = hechos_semana["costo_flete_estimado"] / hechos_semana["valor_factura_pedido"]
    if ratio_flete > umbrales["flete_sobre_factura"]:
        vetos.append(
            f"R4 (VETO) - El flete (${hechos_semana['costo_flete_estimado']:,}) es el "
            f"{ratio_flete:.1%} de la factura (${hechos_semana['valor_factura_pedido']:,}), "
            f"por encima del {umbrales['flete_sobre_factura']:.0%} permitido"
        )

    if not hechos_semana["precio_actualizado_en_punto_de_venta"]:
        vetos.append(
            "R5 (VETO) - El punto de venta no ha actualizado el PVP de la "
            "referencia según la lista vigente de MegaMarket"
        )

    if vetos:
        razones.extend(vetos)
        return {"incluir_en_ruta": False, "prioridad": None, "razones": razones}

    return {"incluir_en_ruta": True, "prioridad": prioridad, "razones": razones}


def decidir_clientes_de_la_ruta(nombre_ruta, base_conocimiento, memoria_trabajo):
    """Corre el motor de inferencia sobre TODOS los clientes de una ruta,
    imprime la traza de razonamiento y devuelve la lista de clientes
    incluidos + su prioridad — el mismo formato que antes recibía
    'clientes_poblado' / 'clientes_valle_aburra' a mano.
    """
    clientes_de_la_ruta = [
        c for c, datos in base_conocimiento["clientes"].items()
        if datos["ruta"] == nombre_ruta
    ]

    incluidos = []
    prioridad_por_cliente = {}

    print(f"--- Motor de inferencia: Ruta {nombre_ruta} ---")
    for cliente in clientes_de_la_ruta:
        conclusion = motor_inferencia(cliente, base_conocimiento, memoria_trabajo)
        estado = "✅ INCLUIR" if conclusion["incluir_en_ruta"] else "❌ excluir"
        extra = f" (prioridad {conclusion['prioridad']})" if conclusion["prioridad"] else ""
        print(f"{cliente} -> {estado}{extra}")
        for razon in conclusion["razones"]:
            print(f"   · {razon}")

        if conclusion["incluir_en_ruta"]:
            incluidos.append(cliente)
            prioridad_por_cliente[cliente] = conclusion["prioridad"]

    print(f">>> Clientes en la ruta de esta semana ({len(incluidos)} de {len(clientes_de_la_ruta)}): {incluidos}\n")
    return incluidos, prioridad_por_cliente

## Parte 5 — Ejecución: sistema experto + A* trabajando juntos

Por cada ruta: primero corre el **motor de inferencia** (decide quién
entra esta semana y con qué prioridad), y su salida se le entrega
directo a `recorrido_completo` del Sprint 1 — sin tocar nada del
algoritmo de búsqueda. Al inicio se imprime la política H1 como
recordatorio del compromiso de servicio bajo el que debe operar la ruta.

In [40]:
# =============================================================================
# 9. EJECUCIÓN
# =============================================================================
REDES = {"Poblado": red_poblado, "Valle de Aburrá": red_valle_aburra}

if __name__ == "__main__":
    # H1 es una política, no una regla de filtrado: solo se documenta aquí.
    horas_min, horas_max = BASE_CONOCIMIENTO["politicas"]["ventana_entrega_horas"]
    print(f"Política de entrega (H1): todo pedido incluido debe entregarse entre "
          f"{horas_min} y {horas_max} horas después de generada la orden.\n")

    for nombre_ruta in ["Poblado", "Valle de Aburrá"]:
        print("#" * 60)
        print(f"RUTA {nombre_ruta.upper()}")
        print("#" * 60)

        # 1) Sistema experto: ¿quién entra en la ruta de esta semana?
        clientes_de_hoy, prioridad = decidir_clientes_de_la_ruta(
            nombre_ruta, BASE_CONOCIMIENTO, MEMORIA_TRABAJO
        )

        # 2) A* (Sprint 1): construir el recorrido con esos clientes
        red = REDES[nombre_ruta]
        paradas, distancia_total, orden_clientes = recorrido_completo(
            red, "Bodega Olaya Herrera", clientes_de_hoy,
            destino_final=BASE_CONOCIMIENTO["destino_final"],
            prioridad=prioridad,
        )

        print("-- Recorrido de esta semana --")
        print(" → ".join(paradas))
        print(f"Distancia total del recorrido: {distancia_total:.1f} km")
        print(f"Orden de visita a clientes: {orden_clientes}")
        print()

Política de entrega (H1): todo pedido incluido debe entregarse entre 48 y 72 horas después de generada la orden.

############################################################
RUTA POBLADO
############################################################
--- Motor de inferencia: Ruta Poblado ---
Fruver del Río -> ❌ excluir
   · R1 - El mercaderista reportó stock bajo en su visita (rotación alta -> prioridad ALTA)
   · R4 (VETO) - El flete ($50,000) es el 5.9% de la factura ($850,000), por encima del 4% permitido
Fruver Visitación -> ❌ excluir
   · R6 - Por defecto: no cumple ningún criterio de inclusión esta semana
Five Sisters -> ✅ INCLUIR (prioridad BAJA)
   · R2 - Sin visita reciente del mercaderista, 18 días sin entrega (> 15), rotación media
Supermercado Merka 10 -> ✅ INCLUIR (prioridad ALTA)
   · R3 - Queja urgente reportada: debe visitarse primero
Hotel Four Points -> ❌ excluir
   · R6 - Por defecto: no cumple ningún criterio de inclusión esta semana
Mercados Verona -> ✅ INCLUIR (prio

---

# Sistema experto 2 — Decisión de negocio: ¿nueva referencia en el portafolio?

El sistema anterior decide **logística** (quién entra en la ruta de esta
semana). Este es un sistema experto **independiente**, para una decisión
distinta: si MegaMarket SAS debe incursionar en distribuir una **nueva
categoría de producto**. Nada de A* ni de heurísticas de ruta acá — es
un sistema experto puro, basado en reglas.

Sigue exactamente la misma arquitectura de tres piezas (base de
conocimiento, memoria de trabajo, motor de inferencia), pero para no
confundir sus reglas con las del sistema de logística, acá las llamamos
**C1 a C7** (de "criterio") en vez de R1, R2, etc.

## Parte 1 — Base de conocimiento

Acá residen las reglas generales del negocio: los 7 criterios que aplica
el gerente general de MegaMarket para evaluar cualquier producto nuevo.
A diferencia del sistema de logística, **acá no hay jerarquía de veto ni
prioridades**: los 7 criterios son requisitos conjuntos — todos deben
cumplirse para que el producto sea viable. Es un chequeo tipo
"todas las luces en verde", no una competencia entre reglas.

**Nota de alcance:** el umbral de margen mínimo (`margen_minimo_pct`) no
venía con un número exacto en las reglas del gerente ("buena
rentabilidad"), así que dejé un valor de referencia (12%) en el
contexto de la base de conocimiento. Igual que el umbral de vida útil,
ya no se usa para *calcular* el criterio en código — ahora es quien
llena el CSV de la memoria de trabajo el que compara el dato real del
producto contra este umbral y marca directamente Sí/No en C6 y C7 (ver
Parte 2). Los umbrales quedan documentados aquí como la política de
referencia contra la que se debe comparar.

In [41]:
# =============================================================================
# 10. BASE DE CONOCIMIENTO — Decisión de portafolio
# =============================================================================
BASE_CONOCIMIENTO_PORTAFOLIO = {
    "contexto": {
        "canal_objetivo": "Cadenas de supermercado (mismo canal que opera MegaMarket)",
        "categoria_actual": "Consumo masivo - alimentos y bebidas",
    },
    "umbrales": {
        "vida_util_minima_meses": 6,   # Criterio 6 del gerente general
        "margen_minimo_pct": 12,       # supuesto de política interna (ajustar con dato real)
    },
}

# Los 7 criterios del gerente general de MegaMarket SAS, formalizados como reglas:
#
#   C1 - SI el producto es afín al portafolio actual (consumo masivo) Y se
#        vende por el mismo canal/segmento (cadenas de supermercado)
#        ENTONCES cumple afinidad de portafolio y canal.
#   C2 - SI el producto tiene registro sanitario vigente Y ficha técnica de
#        respaldo ENTONCES cumple el criterio legal.
#   C3 - SI el producto es fácil de transportar/almacenar/manipular Y NO
#        representa riesgo de contaminación cruzada con los alimentos que ya
#        maneja MegaMarket ENTONCES cumple viabilidad operativa.
#   C4 - SI el producto tiene alta calidad Y presentación estéticamente
#        atractiva ENTONCES cumple el criterio de calidad/presentación.
#   C5 - SI el producto permite venta cruzada con el portafolio actual
#        ENTONCES cumple complementariedad.
#   C6 - SI la vida útil del producto es >= 6 meses ENTONCES cumple vida útil.
#   C7 - SI el producto tiene precio asequible Y un margen de rentabilidad
#        >= al mínimo esperado ENTONCES cumple precio/rentabilidad.
#
# Un producto es VIABLE solo si cumple los 7 al mismo tiempo (C1 AND C2 AND
# ... AND C7). Si falla uno solo, no entra al portafolio.

CRITERIOS_NOMBRES = {
    "C1": "Afinidad de portafolio y canal/segmento",
    "C2": "Cumplimiento legal (registro sanitario y ficha técnica)",
    "C3": "Viabilidad operativa (transporte, almacenamiento, manipulación)",
    "C4": "Estándares de calidad y presentación",
    "C5": "Complementariedad / venta cruzada",
    "C6": "Vida útil (mínimo 6 meses)",
    "C7": "Precio asequible y rentabilidad",
}

# Descripción completa de cada criterio (SI...ENTONCES), para mostrar como
# el detalle de cada fila cuando el motor de inferencia evalúa un producto.
CRITERIOS_DETALLE = {
    "C1": "El producto es afín al portafolio actual (consumo masivo) y se vende por el mismo canal y segmento: cadenas de supermercado.",
    "C2": "El producto cuenta con registro sanitario vigente y ficha técnica de respaldo.",
    "C3": "Es fácil de transportar, almacenar y manipular, y no representa riesgo de contaminación cruzada con los alimentos que ya maneja MegaMarket.",
    "C4": "Cumple altos estándares de calidad y tiene una presentación estéticamente atractiva.",
    "C5": "Complementa el portafolio actual y permite hacer venta cruzada.",
    "C6": f"Tiene una vida útil de {BASE_CONOCIMIENTO_PORTAFOLIO['umbrales']['vida_util_minima_meses']} meses o más.",
    "C7": f"Tiene un precio asequible y un margen de rentabilidad de al menos {BASE_CONOCIMIENTO_PORTAFOLIO['umbrales']['margen_minimo_pct']}%.",
}

## Parte 2 — Memoria de trabajo: las 3 referencias candidatas

Los hechos del **problema actual** ya no se escriben a mano en el
notebook: viven en `memoria_trabajo_portafolio.csv`, con **una fila por
producto candidato y una columna por cada uno de los 7 criterios**
(C1 a C7), marcando Sí o No según si el producto cumple ese requisito.

Es una memoria de trabajo más simple que la del sistema de logística:
en vez de guardar los hechos finos (por ejemplo, la vida útil exacta en
meses) y dejar que el motor calcule si eso pasa el umbral, acá quien
llena el CSV ya resuelve esa comparación al marcar Sí/No en C6 — como lo
haría alguien evaluando el producto con la lista de chequeo del gerente
en la mano. El motor de inferencia se concentra en la parte que sí
requiere razonamiento: verificar que las 7 luces estén en verde a la
vez, y explicar cuáles no lo están cuando no es así.

**Antes de correr la siguiente celda:** sube `memoria_trabajo_portafolio.csv`
a esta sesión de Colab, igual que hiciste con el CSV de logística.

In [42]:
# =============================================================================
# 11. MEMORIA DE TRABAJO — las 3 referencias candidatas (leídas desde CSV)
# =============================================================================
import pandas as pd

memoria_portafolio_df = pd.read_csv("memoria_trabajo_portafolio.csv", encoding="utf-8")

# Reutilizamos a_booleano(), la misma función que ya se definió para el
# sistema de logística: convierte 'Sí' / 'No' del CSV a True / False.
MEMORIA_TRABAJO_PORTAFOLIO = {}
for _, fila in memoria_portafolio_df.iterrows():
    MEMORIA_TRABAJO_PORTAFOLIO[fila["producto"]] = {
        criterio: a_booleano(fila[criterio]) for criterio in CRITERIOS_NOMBRES
    }

# Vista rápida de lo que se cargó, para confirmar que el CSV subió bien
memoria_portafolio_df

KeyError: 'C1'

## Parte 3 — Motor de inferencia

A diferencia del sistema de logística, acá la memoria de trabajo ya trae
el veredicto de cada criterio (Sí/No), así que el motor no deriva nada
de hechos más finos: su razonamiento es verificar que **las 7 luces
estén en verde a la vez** y, si alguna no lo está, explicar cuál —
para poder sustentar el "por qué" caso por caso, tal como lo pediste.

In [ ]:
# =============================================================================
# 12. MOTOR DE INFERENCIA — Decisión de portafolio
# =============================================================================
def motor_inferencia_portafolio(hechos, criterios_nombres, criterios_detalle):
    """Verifica, para UN producto candidato, si cumple cada uno de los 7
    criterios del gerente general, y arma el veredicto final: viable
    solo si los 7 se cumplen a la vez.
    """
    evaluacion = {
        c: {"cumple": hechos[c], "detalle": criterios_detalle[c]}
        for c in criterios_nombres
    }
    incumplidos = [c for c, v in evaluacion.items() if not v["cumple"]]
    viable = len(incumplidos) == 0
    return {"viable": viable, "incumplidos": incumplidos, "evaluacion": evaluacion}


def evaluar_producto(nombre_producto, memoria_trabajo, criterios_nombres, criterios_detalle):
    """Corre el motor de inferencia sobre UN producto candidato e imprime
    la casuística completa: criterio por criterio, y el veredicto final.
    """
    hechos = memoria_trabajo[nombre_producto]
    resultado = motor_inferencia_portafolio(hechos, criterios_nombres, criterios_detalle)

    print(f"=== {nombre_producto} ===")
    for criterio_id, datos in resultado["evaluacion"].items():
        marca = "✅" if datos["cumple"] else "❌"
        print(f"{marca} {criterio_id} ({criterios_nombres[criterio_id]}): {datos['detalle']}")

    if resultado["viable"]:
        print("VEREDICTO: VIABLE. Cumple los 7 criterios del gerente general.\n")
    else:
        nombres_incumplidos = ", ".join(f"{c} ({criterios_nombres[c]})" for c in resultado["incumplidos"])
        print(f"VEREDICTO: NO VIABLE. No cumple: {nombres_incumplidos}.\n")

    return resultado

## Parte 4 — Ejecución: casuística de las 3 referencias

Se evalúa cada producto uno por uno y se cierra con una tabla resumen
comparativa.

In [ ]:
# =============================================================================
# 13. EJECUCIÓN — evaluación de las 3 referencias candidatas
# =============================================================================
if __name__ == "__main__":
    resultados_portafolio = {}
    for producto in MEMORIA_TRABAJO_PORTAFOLIO:
        resultados_portafolio[producto] = evaluar_producto(
            producto, MEMORIA_TRABAJO_PORTAFOLIO, CRITERIOS_NOMBRES, CRITERIOS_DETALLE
        )

    # --- Tabla resumen ---
    print("=" * 78)
    print("RESUMEN COMPARATIVO")
    print("=" * 78)
    encabezado = "Producto".ljust(34) + "".join(c.ljust(5) for c in CRITERIOS_NOMBRES) + "Veredicto"
    print(encabezado)
    for producto, resultado in resultados_portafolio.items():
        fila = producto.ljust(34)
        for c in CRITERIOS_NOMBRES:
            fila += ("OK".ljust(5) if resultado["evaluacion"][c]["cumple"] else "NO".ljust(5))
        fila += "VIABLE" if resultado["viable"] else "NO VIABLE"
        print(fila)

=== Productos de aseo para el hogar ===
❌ C1 (Afinidad de portafolio y canal/segmento): El producto es afín al portafolio actual (consumo masivo) y se vende por el mismo canal y segmento: cadenas de supermercado.
✅ C2 (Cumplimiento legal (registro sanitario y ficha técnica)): El producto cuenta con registro sanitario vigente y ficha técnica de respaldo.
❌ C3 (Viabilidad operativa (transporte, almacenamiento, manipulación)): Es fácil de transportar, almacenar y manipular, y no representa riesgo de contaminación cruzada con los alimentos que ya maneja MegaMarket.
✅ C4 (Estándares de calidad y presentación): Cumple altos estándares de calidad y tiene una presentación estéticamente atractiva.
✅ C5 (Complementariedad / venta cruzada): Complementa el portafolio actual y permite hacer venta cruzada.
✅ C6 (Vida útil (mínimo 6 meses)): Tiene una vida útil de 6 meses o más.
✅ C7 (Precio asequible y rentabilidad): Tiene un precio asequible y un margen de rentabilidad de al menos 12%.
VEREDICTO: N